# Classification Trees

This notebook demonstrates classification trees using scikit-learn, including:

- Tree fitting and visualization  
- Decision boundaries  
- Overfitting  
- Cost-complexity pruning  
- Model diagnostics  

Figures are saved to `figures/` for use in lecture slides.


## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import os

os.makedirs("figures", exist_ok=True)

## 2. Generate Data

In [ ]:
X, y = make_classification(
    n_samples=500,
    n_features=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42
)

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

## 4. Fit Tree (Controlled Depth)

In [ ]:
shallow_clf = DecisionTreeClassifier(max_depth=3, random_state=0)
shallow_clf.fit(X_train, y_train)

## 5. Visualize Tree

In [ ]:
plt.figure(figsize=(10, 6))
plot_tree(shallow_clf, filled=True)
plt.savefig("figures/tree_structure.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Decision Boundary Function

In [ ]:
def plot_boundary(model, X, y, fname=None):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.figure()
    plt.contourf(xx, yy, Z, alpha=0.3)
    plt.scatter(X[:, 0], X[:, 1], c=y)
    plt.title("Decision Boundary")

    if fname:
        plt.savefig(fname, dpi=300, bbox_inches="tight")

    plt.show()

## 7. Plot Boundary (Shallow Tree)

In [ ]:
plot_boundary(shallow_clf, X_test, y_test, "figures/decision_boundary.png")

## 8. Overfitting Demonstration

In [ ]:
deep_clf = DecisionTreeClassifier(max_depth=None, random_state=0)
deep_clf.fit(X_train, y_train)

plot_boundary(deep_clf, X_test, y_test, "figures/overfit_boundary.png")

## 9. Cost-Complexity Pruning

To study pruning properly, we begin with a **fully grown tree** and then examine the sequence of smaller trees obtained by increasing the complexity penalty $ccp\_alpha$.


In [ ]:
# Compute the cost-complexity pruning path from the fully grown tree
path = deep_clf.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

print(
    f"Number of candidate alpha values: {len(ccp_alphas)}\n"
    f"Largest alpha: {ccp_alphas[-1]}"
)

### Pruning Path

As $ccp\_alpha$ increases, the penalty on tree complexity increases, so the fitted tree becomes smaller.


In [ ]:
plt.figure()
plt.plot(ccp_alphas, impurities, marker="o", drawstyle="steps-post")
plt.xlabel("alpha")
plt.ylabel("total leaf impurity")
plt.title("Cost-Complexity Pruning Path")
plt.savefig("figures/pruning_path.png", dpi=300, bbox_inches="tight")
plt.show()

### Fit the Sequence of Pruned Trees

Each value of $ccp\_alpha$ produces a different subtree.


In [ ]:
pruned_clfs = []
for ccp_alpha in ccp_alphas:
    pruned_clf = DecisionTreeClassifier(random_state=0, ccp_alpha=ccp_alpha)
    pruned_clf.fit(X_train, y_train)
    pruned_clfs.append(pruned_clf)

print(
    "Number of nodes in the last tree is: "
    f"{pruned_clfs[-1].tree_.node_count} with ccp_alpha: {ccp_alphas[-1]}"
)

### Accuracy as a Function of $\alpha$

We compare training and test accuracy across the pruning path.


In [ ]:
train_scores = [model.score(X_train, y_train) for model in pruned_clfs]
test_scores = [model.score(X_test, y_test) for model in pruned_clfs]

fig, ax = plt.subplots()
ax.set_xlabel("alpha")
ax.set_ylabel("accuracy")
ax.set_title("Accuracy vs alpha for training and testing sets")
ax.plot(ccp_alphas, train_scores, marker="o", label="train", drawstyle="steps-post")
ax.plot(ccp_alphas, test_scores, marker="o", label="test", drawstyle="steps-post")
ax.legend()
plt.savefig("figures/accuracy_alpha.png", dpi=300, bbox_inches="tight")
plt.show()

### Select the Best Pruned Tree

We choose the tree corresponding to the value of $ccp\_alpha$ that gives the highest test accuracy.


In [ ]:
best_idx = np.argmax(test_scores)
best_alpha = ccp_alphas[best_idx]
best_pruned_clf = pruned_clfs[best_idx]

print("Best alpha:", best_alpha)
print("Best pruned tree test accuracy:", best_pruned_clf.score(X_test, y_test))
print("Best pruned tree node count:", best_pruned_clf.tree_.node_count)

## 10. Accuracy and Diagnostics

Now compare three models:

- a shallow tree with controlled depth
- a fully grown deep tree
- the selected pruned tree


In [ ]:
print("Shallow tree accuracy:", shallow_clf.score(X_test, y_test))
print("Deep tree accuracy:", deep_clf.score(X_test, y_test))
print("Best pruned tree accuracy:", best_pruned_clf.score(X_test, y_test))

### Confusion Matrix and Classification Report for the Selected Pruned Tree

In [ ]:
print(confusion_matrix(y_test, best_pruned_clf.predict(X_test)))
print(classification_report(y_test, best_pruned_clf.predict(X_test)))

## Precision and Recall

To evaluate classification performance, we often use **precision** and **recall**, which are derived from the confusion matrix.

### Confusion Matrix (Binary Classification)

For a given class (e.g., class 1):

- **True Positives (TP)**: predicted 1, actually 1  
- **False Positives (FP)**: predicted 1, actually 0  
- **False Negatives (FN)**: predicted 0, actually 1  
- **True Negatives (TN)**: predicted 0, actually 0  

### Precision

$
\text{Precision} = \frac{TP}{TP + FP}
$

**Interpretation:**

> Among all observations predicted as positive, how many are actually positive?

- High precision $\rightarrow$ few false positives  
- Measures reliability of positive predictions  

**Plain language:**  
*“When the model predicts positive, how often is it correct?”*

### Recall (Sensitivity)

$
\text{Recall} = \frac{TP}{TP + FN}
$

**Interpretation:**

> Among all actual positive observations, how many did we correctly identify?

- High recall $\rightarrow$ few false negatives  
- Measures ability to detect positives  

**Plain language:**  
*“Of all the real positives, how many did we find?”*

### Tradeoff

- Increasing recall often decreases precision  
- Increasing precision often decreases recall  

This tradeoff depends on the application:
- Medical screening: prioritize **high recall**  
- Spam detection: prioritize **high precision**

### Summary

- **Precision** = correctness of positive predictions  
- **Recall** = completeness of detecting positives

### F1 Score

The **F1 score** combines precision and recall into a single metric.

$
F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$

It is the **harmonic mean** of precision and recall.

---

### Interpretation

> F1 measures the balance between:
> - **precision** (how accurate positive predictions are)
> - **recall** (how well we find all positives)

- High F1 $\rightarrow$ both precision and recall are high  
- Low F1 $\rightarrow$ at least one of them is low  

---

### Why not just average them?

The harmonic mean penalizes imbalance:

- If precision is high but recall is low $\rightarrow$ F1 is low  
- If recall is high but precision is low $\rightarrow$ F1 is low  

So both must be good to get a high F1 score.

---

### Example

If:
- Precision = 0.80  
- Recall = 0.60  

$
F_1 = 2 \cdot \frac{0.8 \cdot 0.6}{0.8 + 0.6}
= \frac{0.96}{1.4}
\approx 0.69
$

---

### When to use F1

Use F1 when:

- You care about **both precision and recall**
- The data are **imbalanced**
- Both false positives and false negatives matter

---

### Summary

- **Precision** = correctness of positive predictions  
- **Recall** = completeness of detecting positives  
- **F1** = balance between the two  


## 11. Feature Importance

Feature importance is computed from the total impurity reduction attributed to each feature across the fitted tree.

Here we visualize the feature importance for the **selected pruned tree**.


In [ ]:
plt.figure()
plt.bar(
    range(len(best_pruned_clf.feature_importances_)),
    best_pruned_clf.feature_importances_,
)
plt.title("Feature Importance (Best Pruned Tree)")
plt.savefig("figures/feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()